# 03 — Statistical Significance TestingRepeats the binary random-vs-temporal audit across 10 random seeds per model (Random Forest, LightGBM, XGBoost), then computes paired Wilcoxon signed-rank tests, paired t-tests, and 95% confidence intervals on the inflation. Produces Table 4 (columns: mean performance, 95% CI, Wilcoxon p) and saves `significance_test_raw_runs.csv`, consumed by `04_cliffs_delta.ipynb`.**Note:** the MLP is excluded from this repeated-runs analysis due to its substantially higher per-run training cost (see paper Section 5.2 for justification).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from scipy.stats import wilcoxon, ttest_rel
from collections import Counter
import gc, warnings
warnings.filterwarnings('ignore')

SAVE_PATH    = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed'
LABEL_COL    = 'Label'
ordered_days = ['monday','tuesday','wednesday','thursday','friday']
N_RUNS       = 10                       # 10 repeated runs per reviewer recommendation
SEEDS        = list(range(42, 42 + N_RUNS))

CATEGORICAL_EXCLUDE = ['Destination Port', 'Source Port']

if 'day_dfs' not in dir() or len(day_dfs) == 0:
    day_dfs = {}
    for day in ordered_days:
        path = SAVE_PATH + f'/{day}_preprocessed.csv'
        if not os.path.exists(path): continue
        df        = pd.read_csv(path, low_memory=False)
        df['day'] = day
        day_dfs[day] = df
    print(f'Loaded {sum(len(v) for v in day_dfs.values()):,} rows')


# ══════════════════════════════════════════════════════════════
#  PREPROCESSING — parameterised by seed
# ══════════════════════════════════════════════════════════════
def preprocess_binary(df_tr, df_te, seed, label_col=LABEL_COL,
                      variance_thresh=0.01, corr_thresh=0.95):
    try:
        X_tr     = df_tr.drop(columns=[label_col,'day'], errors='ignore').copy()
        X_te     = df_te.drop(columns=[label_col,'day'], errors='ignore').copy()
        y_tr_raw = df_tr[label_col].copy()
        y_te_raw = df_te[label_col].copy()

        port_cols = [c for c in X_tr.columns if c in CATEGORICAL_EXCLUDE]
        X_tr.drop(columns=port_cols, inplace=True, errors='ignore')
        X_te.drop(columns=port_cols, inplace=True, errors='ignore')

        X_tr.drop(columns=X_tr.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_te.drop(columns=X_te.select_dtypes(exclude=[np.number]).columns, inplace=True)
        X_tr.replace([np.inf,-np.inf], np.nan, inplace=True)
        X_te.replace([np.inf,-np.inf], np.nan, inplace=True)
        mask_tr  = X_tr.notna().all(axis=1)
        mask_te  = X_te.notna().all(axis=1)
        X_tr     = X_tr[mask_tr].reset_index(drop=True)
        y_tr_raw = y_tr_raw[mask_tr].reset_index(drop=True)
        X_te     = X_te[mask_te].reset_index(drop=True)
        y_te_raw = y_te_raw[mask_te].reset_index(drop=True)

        common = [c for c in X_tr.columns if c in X_te.columns]
        X_tr = X_tr[common].copy(); X_te = X_te[common].copy()

        vt   = VarianceThreshold(threshold=variance_thresh)
        arr  = vt.fit_transform(X_tr)
        cols = np.array(common)[vt.get_support()]
        X_tr = pd.DataFrame(arr, columns=cols)
        X_te = pd.DataFrame(vt.transform(X_te), columns=cols)

        corr  = X_tr.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop  = [c for c in upper.columns if any(upper[c] >= corr_thresh)]
        X_tr.drop(columns=drop, inplace=True)
        X_te.drop(columns=drop, inplace=True, errors='ignore')
        feature_names = X_tr.columns.tolist()

        scaler  = StandardScaler()
        X_tr_sc = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_names)
        X_te_sc = pd.DataFrame(scaler.transform(X_te),     columns=feature_names)

        le = LabelEncoder()
        le.fit(['BENIGN','ATTACK'])
        y_tr_raw = y_tr_raw.where(y_tr_raw.isin(['BENIGN','ATTACK']), 'ATTACK')
        y_te_raw = y_te_raw.where(y_te_raw.isin(['BENIGN','ATTACK']), 'ATTACK')
        y_tr = pd.Series(le.transform(y_tr_raw), name='label')
        y_te = pd.Series(le.transform(y_te_raw), name='label')

        counts  = Counter(y_tr)
        min_cls = min(counts, key=counts.get)
        maj_cls = max(counts, key=counts.get)
        min_cnt = counts[min_cls]
        maj_cnt = counts[maj_cls]
        k       = max(1, min(5, min_cnt - 1))
        target  = max(min_cnt, min(30_000, maj_cnt))
        strat   = {min_cls: target}

        smote = SMOTE(random_state=seed, k_neighbors=k, sampling_strategy=strat)
        X_sm, y_sm = smote.fit_resample(X_tr_sc, y_tr)
        return X_sm, y_sm, X_te_sc, y_te, le

    except Exception as e:
        import traceback
        print(f'  [ERROR]: {e}')
        traceback.print_exc()
        return (None,)*5


def evaluate(model, X_te, y_te, le):
    preds = model.predict(X_te)
    classes    = list(le.classes_)
    benign_idx = classes.index('BENIGN')
    attack_idx = classes.index('ATTACK')

    acc = accuracy_score(y_te, preds)
    f1m = f1_score(y_te, preds, average='macro', zero_division=0)

    atk_mask   = (y_te == attack_idx)
    atk_recall = (preds[atk_mask] == attack_idx).mean() if atk_mask.any() else 0.0

    cm  = confusion_matrix(y_te, preds, labels=[benign_idx, attack_idx])
    TN, FP = cm[0,0], cm[0,1]
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    try:
        proba = model.predict_proba(X_te)[:, attack_idx]
        roc   = roc_auc_score((y_te == attack_idx).astype(int), proba)
    except Exception:
        roc = float('nan')

    return {'accuracy': acc, 'f1_macro': f1m, 'attack_recall': atk_recall,
            'FPR': FPR, 'ROC_AUC': roc}


# ══════════════════════════════════════════════════════════════
#  REPEATED-RUNS AUDIT
# ══════════════════════════════════════════════════════════════
models_fn = {
    'RandomForest': lambda seed: RandomForestClassifier(
        n_estimators=100, max_depth=10, class_weight='balanced',
        random_state=seed, n_jobs=-1),
    'LightGBM': lambda seed: lgb.LGBMClassifier(
        n_estimators=100, max_depth=8, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, class_weight='balanced',
        random_state=seed, n_jobs=-1, verbose=-1),
    'XGBoost': lambda seed: xgb.XGBClassifier(
        n_estimators=100, max_depth=6, subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=seed, n_jobs=-1, verbosity=0),
}

all_results = []  # one row per (model, split, seed, metric values)

print('='*70)
print(f'REPEATED-RUNS SIGNIFICANCE TEST  ({N_RUNS} seeds per model per split)')
print('='*70)

for run_idx, seed in enumerate(SEEDS):
    print(f'\n--- Run {run_idx+1}/{N_RUNS}  (seed={seed}) ---')

    # ── temporal split (Mon-Thu train, Fri test) — only the train-side
    # sampling/SMOTE changes with seed; the day boundary stays fixed ──
    df_tr_raw = pd.concat(
        [day_dfs[d] for d in ['monday','tuesday','wednesday','thursday']],
        ignore_index=True)
    df_te_raw = day_dfs['friday'].copy()
    for df in [df_tr_raw, df_te_raw]:
        df[LABEL_COL] = df[LABEL_COL].apply(
            lambda x: 'BENIGN' if x=='BENIGN' else 'ATTACK')

    if len(df_tr_raw) > 200_000:
        df_tr_raw = (df_tr_raw.groupby(LABEL_COL, group_keys=False)
                     .apply(lambda x: x.sample(
                         min(len(x), int(200_000*len(x)/len(df_tr_raw))),
                         random_state=seed))
                     .reset_index(drop=True))
    if len(df_te_raw) > 100_000:
        df_te_raw = (df_te_raw.groupby(LABEL_COL, group_keys=False)
                     .apply(lambda x: x.sample(
                         min(len(x), int(100_000*len(x)/len(df_te_raw))),
                         random_state=seed))
                     .reset_index(drop=True))

    result_temp = preprocess_binary(df_tr_raw, df_te_raw, seed)
    if result_temp[0] is None:
        print('  [SKIP] temporal preprocessing failed'); continue
    X_tr_temp, y_tr_temp, X_te_temp, y_te_temp, le_temp = result_temp
    del df_tr_raw, df_te_raw; gc.collect()

    # ── random split — re-sampled fresh each run with the new seed ──
    df_all = pd.concat(day_dfs.values(), ignore_index=True)
    df_all[LABEL_COL] = df_all[LABEL_COL].apply(
        lambda x: 'BENIGN' if x=='BENIGN' else 'ATTACK')
    if len(df_all) > 200_000:
        df_all = (df_all.groupby(LABEL_COL, group_keys=False)
                  .apply(lambda x: x.sample(
                      min(len(x), int(200_000*len(x)/len(df_all))),
                      random_state=seed))
                  .reset_index(drop=True))
    X_all = df_all.drop(columns=[LABEL_COL,'day'], errors='ignore')
    y_all = df_all[LABEL_COL]
    X_r_tr, X_r_te, y_r_tr, y_r_te = train_test_split(
        X_all, y_all, test_size=0.2, random_state=seed, stratify=y_all)
    df_r_tr = pd.concat([X_r_tr, y_r_tr], axis=1)
    df_r_te = pd.concat([X_r_te, y_r_te], axis=1)

    result_rand = preprocess_binary(df_r_tr, df_r_te, seed)
    if result_rand[0] is None:
        print('  [SKIP] random preprocessing failed'); continue
    X_tr_rand, y_tr_rand, X_te_rand, y_te_rand, le_rand = result_rand
    del df_all, df_r_tr, df_r_te, X_all, y_all; gc.collect()

    for model_name, model_fn in models_fn.items():
        model = model_fn(seed)
        model.fit(X_tr_rand, y_tr_rand)
        metrics_rand = evaluate(model, X_te_rand, y_te_rand, le_rand)

        model = model_fn(seed)
        model.fit(X_tr_temp, y_tr_temp)
        metrics_temp = evaluate(model, X_te_temp, y_te_temp, le_temp)

        all_results.append({
            'model': model_name, 'seed': seed,
            'acc_random': metrics_rand['accuracy'],
            'acc_temporal': metrics_temp['accuracy'],
            'f1m_random': metrics_rand['f1_macro'],
            'f1m_temporal': metrics_temp['f1_macro'],
            'recall_random': metrics_rand['attack_recall'],
            'recall_temporal': metrics_temp['attack_recall'],
        })
        print(f'  {model_name:14s} | Acc R:{metrics_rand["accuracy"]:.4f} '
              f'T:{metrics_temp["accuracy"]:.4f} | F1m R:{metrics_rand["f1_macro"]:.4f} '
              f'T:{metrics_temp["f1_macro"]:.4f}')
        gc.collect()


# ══════════════════════════════════════════════════════════════
#  STATISTICAL SIGNIFICANCE ANALYSIS
# ══════════════════════════════════════════════════════════════
df_runs = pd.DataFrame(all_results)
df_runs.to_csv(SAVE_PATH + '/significance_test_raw_runs.csv', index=False)
print(f'\nSaved raw runs → significance_test_raw_runs.csv')

print('\n' + '='*78)
print('STATISTICAL SIGNIFICANCE OF TEMPORAL LEAKAGE INFLATION')
print(f'(n={N_RUNS} paired runs per model, Wilcoxon signed-rank + paired t-test)')
print('='*78)

summary_rows = []
for model_name in models_fn.keys():
    sub = df_runs[df_runs.model == model_name]
    if len(sub) < 2:
        print(f'\n{model_name}: insufficient runs, skipping')
        continue

    print(f'\n--- {model_name} (n={len(sub)} runs) ---')

    for metric in ['acc', 'f1m', 'recall']:
        rand_col = f'{metric}_random'
        temp_col = f'{metric}_temporal'
        rand_vals = sub[rand_col].values
        temp_vals = sub[temp_col].values
        diffs     = rand_vals - temp_vals

        mean_rand = rand_vals.mean()
        mean_temp = temp_vals.mean()
        mean_diff = diffs.mean()
        std_diff  = diffs.std(ddof=1)
        se        = std_diff / np.sqrt(len(diffs))
        ci95_lo   = mean_diff - 1.96 * se
        ci95_hi   = mean_diff + 1.96 * se

        # paired t-test
        t_stat, t_p = ttest_rel(rand_vals, temp_vals)

        # Wilcoxon signed-rank — robust to non-normality, small n
        try:
            w_stat, w_p = wilcoxon(rand_vals, temp_vals)
        except ValueError:
            w_stat, w_p = float('nan'), float('nan')

        metric_label = {'acc':'Accuracy', 'f1m':'F1-Macro', 'recall':'Attack Recall'}[metric]
        print(f'  {metric_label:14s} | Random: {mean_rand:.4f} (±{rand_vals.std(ddof=1):.4f}) | '
              f'Temporal: {mean_temp:.4f} (±{temp_vals.std(ddof=1):.4f})')
        print(f'    Mean inflation: {mean_diff:+.4f}  95% CI: [{ci95_lo:+.4f}, {ci95_hi:+.4f}]')
        print(f'    Paired t-test: t={t_stat:.3f}, p={t_p:.6f}'
              f'  {"***" if t_p<0.001 else "**" if t_p<0.01 else "*" if t_p<0.05 else "ns"}')
        print(f'    Wilcoxon signed-rank: W={w_stat:.1f}, p={w_p:.6f}'
              f'  {"***" if w_p<0.001 else "**" if w_p<0.01 else "*" if w_p<0.05 else "ns"}')

        summary_rows.append({
            'model': model_name, 'metric': metric_label,
            'mean_random': round(mean_rand,4), 'mean_temporal': round(mean_temp,4),
            'mean_inflation_pp': round(mean_diff*100,2),
            'ci95_lo_pp': round(ci95_lo*100,2), 'ci95_hi_pp': round(ci95_hi*100,2),
            't_stat': round(t_stat,3), 't_pvalue': t_p,
            'wilcoxon_stat': round(w_stat,1) if not np.isnan(w_stat) else None,
            'wilcoxon_pvalue': w_p,
        })

summary_df = pd.DataFrame(summary_rows)
print('\n\n' + '='*78)
print('TABLE — SUMMARY WITH SIGNIFICANCE (FOR PAPER)')
print('='*78)
print(summary_df.to_string(index=False))

summary_df.to_csv(SAVE_PATH + '/significance_test_summary.csv', index=False)
print(f'\nSaved → significance_test_summary.csv')

Loaded 2,827,876 rows
REPEATED-RUNS SIGNIFICANCE TEST  (10 seeds per model per split)

--- Run 1/10  (seed=42) ---
  RandomForest   | Acc R:0.9932 T:0.5889 | F1m R:0.9894 T:0.3763
  LightGBM       | Acc R:0.9984 T:0.5891 | F1m R:0.9975 T:0.3717
  XGBoost        | Acc R:0.9988 T:0.6078 | F1m R:0.9981 T:0.4191

--- Run 2/10  (seed=43) ---
  RandomForest   | Acc R:0.9947 T:0.6147 | F1m R:0.9916 T:0.4409
  LightGBM       | Acc R:0.9984 T:0.5947 | F1m R:0.9976 T:0.3865
  XGBoost        | Acc R:0.9988 T:0.6033 | F1m R:0.9981 T:0.4079

--- Run 3/10  (seed=44) ---
  RandomForest   | Acc R:0.9880 T:0.5859 | F1m R:0.9813 T:0.3696
  LightGBM       | Acc R:0.9985 T:0.5893 | F1m R:0.9976 T:0.3722
  XGBoost        | Acc R:0.9989 T:0.5892 | F1m R:0.9982 T:0.3715

--- Run 4/10  (seed=45) ---
  RandomForest   | Acc R:0.9943 T:0.6155 | F1m R:0.9911 T:0.4427
  LightGBM       | Acc R:0.9984 T:0.6112 | F1m R:0.9975 T:0.4278
  XGBoost        | Acc R:0.9988 T:0.5976 | F1m R:0.9981 T:0.3934

--- Run 5/10  (se